Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [1]:
!pip install pyspark

In [11]:
!wget -O owid-covid-data.csv https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv

--2026-05-19 21:32:23--  https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 98391483 (94M) [text/plain]
Saving to: ‘owid-covid-data.csv’

owid-covid-data.csv 100%[===================>]  93.83M   200MB/s    in 0.5s    

2026-05-19 21:32:24 (200 MB/s) - ‘owid-covid-data.csv’ saved [98391483/98391483]



In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Spark session
spark = SparkSession.builder.appName("COVID Analysis").getOrCreate()

# Load dataset
df = spark.read.csv("owid-covid-data.csv", header=True, inferSchema=True)

# Show sample data
df.show(5)

# 4. Pakistan-specific data

df.filter(col("location") == "Pakistan") \
  .select("date", "total_cases", "total_deaths") \
  .show(10)

# 5. Total cases & deaths (latest approx per country)

df.groupBy("location") \
  .max("total_cases", "total_deaths") \
  .show(10)

# 6. Top countries by cases

df.groupBy("location") \
  .max("total_cases") \
  .orderBy(col("max(total_cases)").desc()) \
  .show(10)

# 7. Clean Insights (for report)

print("\nINSIGHTS:")
print("1. USA, India, Brazil have highest COVID-19 cases globally.")
print("2. Pakistan has comparatively lower total cases.")
print("3. Death counts vary across countries.")
print("4. COVID spread is uneven across the world.")


df_clean = df.filter(
    (col("continent").isNotNull()) &
    (~col("location").isin([
        "World",
        "Asia",
        "Europe",
        "European Union (27)",
        "High-income countries",
        "Upper-middle-income countries",
        "Lower-middle-income countries"
    ]))
)

+--------+---------+-----------+----------+-----------+---------+------------------+------------+----------+-------------------+-----------------------+---------------------+------------------------------+------------------------+----------------------+-------------------------------+-----------------+------------+------------------------+-------------+-------------------------+---------------------+---------------------------------+----------------------+----------------------------------+-----------+---------+------------------------+----------------------+------------------+-------------------------------+-------------+--------------+-----------+------------------+-----------------+-----------------------+--------------+----------------+-------------------------+------------------------------+-----------------------------+-----------------------------------+--------------------------+-------------------------------------+------------------------------+-------------------------------